# Filter Bad Images

Scans **all** training images and flags two kinds of bad samples:
1. **Cropped / rim-only** — aspect ratio is far from square
2. **Multi-coin** — Hough circle detection finds 2+ coins in the frame

Run cells 1–4 to build `flagged_df`, then use the galleries to verify before any deletion.

In [13]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

In [47]:
DATA_DIR   = Path('/home/david/coin/FOR_TRAINNING')
OUTPUT_DIR = Path('/home/david/coin/misclassified')
CACHE_CSV  = OUTPUT_DIR / 'image_ar_cache.csv'

# Images much taller than wide → badly cropped rim → DISCARD
AR_DISCARD_MAX = 0.60

# Images moderately wider than tall → both sides of coin side-by-side → SPLIT (keep left half)
AR_SPLIT_MIN   = 1.55
AR_SPLIT_MAX   = 2.50   # above this → too extreme, also badly cropped → DISCARD

In [48]:
if CACHE_CSV.exists():
    print(f'Loading cached aspect ratios from {CACHE_CSV}')
    metrics_df = pd.read_csv(CACHE_CSV)
    print(f'{len(metrics_df)} images loaded from cache.')
else:
    all_images = sorted(DATA_DIR.glob('*/side_a/*.jpg'))
    print(f'Found {len(all_images)} images — reading sizes (no decoding, should be fast)…')

    records = []
    for fp in tqdm(all_images):
        w, h = Image.open(fp).size   # reads JPEG header only, no decode
        records.append({'filepath': str(fp), 'aspect_ratio': w / h})

    metrics_df = pd.DataFrame(records)
    metrics_df.to_csv(CACHE_CSV, index=False)
    print(f'Done. Saved to {CACHE_CSV}')

Found 62134 images — reading sizes (no decoding, should be fast)…


  0%|          | 0/62134 [00:00<?, ?it/s]

Done. Saved to /home/david/coin/misclassified/image_ar_cache.csv


In [49]:
ar = metrics_df['aspect_ratio']

discard_tall_mask = ar < AR_DISCARD_MAX
discard_wide_mask = ar > AR_SPLIT_MAX
split_mask        = (ar >= AR_SPLIT_MIN) & (ar <= AR_SPLIT_MAX)
keep_mask         = ~discard_tall_mask & ~discard_wide_mask & ~split_mask

discard_tall_df = metrics_df[discard_tall_mask].reset_index(drop=True)
discard_wide_df = metrics_df[discard_wide_mask].reset_index(drop=True)
split_df        = metrics_df[split_mask].reset_index(drop=True)
keep_df         = metrics_df[keep_mask].reset_index(drop=True)

print(f'Total images:                         {len(metrics_df)}')
print(f'Discard tall  (AR < {AR_DISCARD_MAX}):      {len(discard_tall_df)}')
print(f'Discard wide  (AR > {AR_SPLIT_MAX}):      {len(discard_wide_df)}')
print(f'Split         ({AR_SPLIT_MIN} <= AR <= {AR_SPLIT_MAX}): {len(split_df)}')
print(f'Keep          (normal):               {len(keep_df)}')

Total images:                         62134
Discard tall  (AR < 0.6):      0
Discard wide  (AR > 2.5):      0
Split         (1.55 <= AR <= 2.5): 0
Keep          (normal):               62134


## Galleries — verify before any action

- **Discard gallery**: tall/narrow images → should look like a clipped coin edge. These will be deleted.
- **Split gallery**: wide images → should show obverse on the left, reverse on the right. The red line shows where the cut will happen.

In [50]:
def make_gallery(df, title_prefix, cols=5, per_page=20, draw_split=False):
    if len(df) == 0:
        print('Nothing to show.')
        return

    n_pages = max(1, math.ceil(len(df) / per_page))
    page_sl = widgets.IntSlider(value=0, min=0, max=n_pages - 1,
                                description='Page:', layout=widgets.Layout(width='350px'))
    cols_sl = widgets.IntSlider(value=cols, min=2, max=8,
                                description='Columns:', layout=widgets.Layout(width='280px'))
    out = widgets.Output()

    def refresh(change=None):
        page      = page_sl.value
        n_cols    = cols_sl.value
        page_data = df.iloc[page * per_page : (page + 1) * per_page]
        n_rows    = math.ceil(len(page_data) / n_cols)

        fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.0, n_rows * 3.2))
        axes = np.array(axes).reshape(-1)
        for ax in axes:
            ax.axis('off')

        for i, (_, row) in enumerate(page_data.iterrows()):
            img = Image.open(row['filepath']).convert('RGB')
            axes[i].imshow(img)
            if draw_split:
                axes[i].axvline(x=img.width / 2, color='red', linewidth=2)
            axes[i].set_title(f"AR={row['aspect_ratio']:.2f}", fontsize=7)
            axes[i].axis('off')

        fig.suptitle(
            f'{title_prefix}  ({len(df)} total)  |  page {page + 1} / {n_pages}',
            fontsize=11, fontweight='bold'
        )
        plt.tight_layout()
        with out:
            clear_output(wait=True)
            plt.show()

    page_sl.observe(refresh, names='value')
    cols_sl.observe(refresh, names='value')
    refresh()
    display(widgets.VBox([widgets.HBox([page_sl, cols_sl]), out]))


print(f'=== DISCARD — too tall / badly cropped ({len(discard_tall_df)}) ===')
make_gallery(discard_tall_df, 'Discard: too tall')

=== DISCARD — too tall / badly cropped (0) ===
Nothing to show.


In [51]:
print(f'=== SPLIT — both sides of coin ({len(split_df)}) ===')
print('Red line shows where the cut happens. Left half (emperor side) will be kept.')
make_gallery(split_df, 'Split: both sides of coin', draw_split=True)

=== SPLIT — both sides of coin (0) ===
Red line shows where the cut happens. Left half (emperor side) will be kept.
Nothing to show.


In [52]:
print(f'=== DISCARD — too wide / badly cropped ({len(discard_wide_df)}) ===')
make_gallery(discard_wide_df, 'Discard: too wide')

=== DISCARD — too wide / badly cropped (0) ===
Nothing to show.


In [46]:
# ── Apply changes to FOR_TRAINNING ────────────────────────────────────────────
# DELETE: discard_tall_df + discard_wide_df
# CROP:   split_df → overwrite with left half in place

import os

deleted = 0
for fp in discard_tall_df['filepath'].tolist() + discard_wide_df['filepath'].tolist():
    try:
        os.remove(fp)
        deleted += 1
    except FileNotFoundError:
        pass

cropped = 0
for fp in split_df['filepath']:
    img = Image.open(fp)
    left_half = img.crop((0, 0, img.width // 2, img.height))
    left_half.save(fp)
    cropped += 1

print(f'Deleted: {deleted}  |  Cropped in place: {cropped}')


Deleted: 230  |  Cropped in place: 299
